## Wine Quality Prediction ? Machine Learning

Author:Claudia Magliano
Date: 2024-03-26


Here we will predict the quality of wine on the basis of given features. We use the wine quality dataset available on Internet for free (Kaggle). This dataset has the fundamental features which are responsible for affecting the quality of the wine. By the use of several Machine learning models, we will predict the quality of the wine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import metrics
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')

In [ ]:

df = pd.read_csv('winequalityN.csv')
print(df.head())

Let?s explore the type of data present in each of the columns present in the dataset.

In [ ]:
df.info()

In [ ]:
df.describe().T


In [ ]:
df.isnull().sum()


Let?s imput the missing values by means as the data present in the different columns are continuous values.

In [ ]:
for col in df.columns:
  if df[col].isnull().sum() > 0:
	  df[col] = df[col].fillna(df[col].mean())

df.isnull().sum().sum()


Let?s draw the histogram to visualise the distribution of the data with continuous values in the columns of the dataset.

In [ ]:
df.hist(bins=20, figsize=(10, 10))
plt.show()

Now let?s draw the count plot to visualise the number data for each quality of wine.



In [ ]:
plt.bar(df['quality'], df['alcohol'])
plt.xlabel('quality')
plt.ylabel('alcohol')
plt.show()

There are times the data provided to us contains redundant features they do not help with increasing the model?s performance that is why we remove them before using them to train our model.

In [ ]:
plt.figure(figsize=(12, 12))
sb.heatmap(df.corr() > 0.7, annot=True, cbar=False)
plt.show()

From the above heat map we can conclude that the ?total sulphur dioxide? and ?free sulphur dioxide? are highly correlated features so, we will remove them.

In [ ]:
df = df.drop('total sulfur dioxide', axis=1)

# Model Development
Let?s prepare our data for training and splitting it into training and validation data so, that we can select which model?s performance is best as per the use case. We will train some of the state of the art machine learning classification models and then select best out of them using validation data.

In [ ]:
df['best quality'] = [1 if x > 5 else 0 for x in df.quality]

In [ ]:
df.replace({'white': 1, 'red': 0}, inplace=True)

After segregating features and the target variable from the dataset we will split it into 80:20 ratio for model selection.

In [ ]:
features = df.drop(['quality', 'best quality'], axis=1)
target = df['best quality']

xtrain, xtest, ytrain, ytest = train_test_split(
	features, target, test_size=0.2, random_state=40)

xtrain.shape, xtest.shape

Normalising the data before training help us to achieve stable and fast training of the model.

Normalization : It is the methodology of arranging a data model to capably store data in an information base. The completed impact is that tedious data is cleared out, and just data related to the attribute is taken care of inside the table.

In [ ]:
norm = MinMaxScaler()
xtrain = norm.fit_transform(xtrain)
xtest = norm.transform(xtest)

As the data has been prepared completely let?s train some state of the art machine learning model on it.

In [ ]:
models = [LogisticRegression(), XGBClassifier(), SVC(kernel='rbf')]

for i in range(3):
	models[i].fit(xtrain, ytrain)

	print(f'{models[i]} : ')
	print('Training Accuracy : ', metrics.roc_auc_score(ytrain, models[i].predict(xtrain)))
	print('Validation Accuracy : ', metrics.roc_auc_score(
		ytest, models[i].predict(xtest)))
	print()

# Model Evaluation
From the above accuracies we can say that Logistic Regression and SVC() classifier performing better on the validation data with less difference between the validation and training data. Let?s plot the confusion matrix as well for the validation data using the Logistic Regression model.

In [ ]:
metrics.confusion_matrix(xtest, ytest)
plt.show()

Let?s also print the classification report for the best performing model.

In [ ]:
print(metrics.classification_report(ytest,
									models[1].predict(xtest)))